# Lab — Build a causal transformer from scratch

**Session 1**

Build a decoder-only language model progressively: one causal head, multi-head attention, a decoder block, then a tiny language model. At every stage, verify both parameter count and causality.


## Lab rules

- Use PyTorch primitives only; do not use `nn.MultiheadAttention` or `nn.Transformer`.
- Preserve shapes explicitly: inputs and block outputs use `(B, T, C)`.
- Apply the lower-triangular causal mask to scores **before** softmax.
- Do not continue until the stage parameter count and causal test pass.


## Tokenization and embeddings


In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer

MODEL_NAME = "distilbert-base-uncased"  # small: 6 layers, 768 dims, runs on CPU

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME, output_hidden_states=True)

In [ ]:

def embed(text: str) -> dict:
    """Text in -> tokens + embeddings out."""
    enc = tokenizer(text, return_tensors="pt")
    input_ids = enc["input_ids"]                       # (1, seq_len)
    tokens = tokenizer.convert_ids_to_tokens(input_ids[0])

    with torch.no_grad():
        out = model(**enc)

        # 1) Static embeddings: pure table lookup, same vector for a token
        #    whatever the sentence around it.
        static = model.embeddings.word_embeddings(input_ids)[0]      # (seq_len, dim)

        # 2) Position embeddings: one learned vector per position 0..seq_len-1.
        positions = torch.arange(input_ids.size(1)).unsqueeze(0)
        pos = model.embeddings.position_embeddings(positions)[0]     # (seq_len, dim)

    # 3) Contextual embeddings: hidden_states[0] is the input to layer 1
    #    (static + position, then LayerNorm); hidden_states[i] is the output
    #    of layer i.
    layers = [h[0] for h in out.hidden_states]         # list of (seq_len, dim)

    return {
        "tokens": tokens,
        "ids": input_ids[0].tolist(),
        "static": static,
        "position": pos,
        "layers": layers,
        "final": out.last_hidden_state[0],
    }


def show(text: str) -> None:
    r = embed(text)
    print(f"Text: {text!r}")
    print(f"Number of tokens: {len(r['tokens'])}")
    print(f"Embedding dimension: {r['static'].shape[1]}")
    print(f"Number of hidden-state tensors (embeddings + layers): {len(r['layers'])}\n")
    print(f"{'pos':>3}  {'token':<12} {'id':>6}  first 4 dims of final embedding")
    for i, (tok, tid) in enumerate(zip(r["tokens"], r["ids"])):
        vec = r["final"][i][:4].tolist()
        print(f"{i:>3}  {tok:<12} {tid:>6}  {[round(v, 3) for v in vec]}")


def word_vector(text: str, word: str, layer: int = -1) -> torch.Tensor:
    """Embedding of the first occurrence of `word` in `text`, at a given layer."""
    r = embed(text)
    idx = r["tokens"].index(word)
    return r["layers"][layer][idx]

### Part 1: Tokenization


In [ ]:

show("Transformers are unbelievably powerful!")

### Part 2: static vs contextual


In [ ]:

s1 = "I deposited money at the bank."
s2 = "We had a picnic on the river bank."
s3 = "I work as an assistant to the regional bank manager"

r1, r2, r3 = embed(s1), embed(s2), embed(s3)
static_bank = r1["static"][r1["tokens"].index("bank")]
static_bank2 = r2["static"][r2["tokens"].index("bank")]
static_bank3 = r3["static"][r3["tokens"].index("bank")]
ctx_bank = word_vector(s1, "bank", layer=-1)
ctx_bank2 = word_vector(s2, "bank", layer=-1)
ctx_bank3 = word_vector(s3, "bank", layer=-1)

cos = lambda a, b: F.cosine_similarity(a, b, dim=0).item()
print("\nCosine similarity between the two 'bank' vectors")
print(f"  static     : {cos(static_bank, static_bank2):.3f}  (should be exactly 1.0)")
print(f"  contextual : {cos(ctx_bank, ctx_bank2):.3f}  (< 1.0: context changed the vector)")
print(f"  contextual : {cos(ctx_bank, ctx_bank3):.3f}  (< 1.0: should be higher than with s2)")


#### Exercises

* Tokenize a rare word, a typo, a number, an emoji, a French sentence.
   What happens? Why does the '##' prefix exist?

* Plot cos(bank_s1, bank_s2) at each layer (loop over layer=0..6).
   At which layer does the model "disambiguate" the word?

* Swap MODEL_NAME for "bert-base-multilingual-cased" and compare the
   tokenization of a French sentence.

* (Bonus) Reduce the contextual vectors of a paragraph to 2D with PCA
   and plot them. Do syntactically similar words cluster?

## Transformer architecture

In [ ]:
import math, random
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu" # or "mps" if you have a compatible macOs device (M1, M2, ...)

C, H, MAX_T, V = 32, 4, 16, tokenizer.vocab_size
D = C // H
assert C % H == 0
print({"device": DEVICE, "C": C, "H": H, "D": D, "MAX_T": MAX_T, "V": V})


### Toy corpus

The corpus is intentionally tiny. It lets you train and generate simple sentence continuations after the architecture passes its tests.


In [ ]:
toy_sentences = [
    "the cat sat .", "the cat ate fish .",
    "the dog sat .", "the dog ate food .",
    "the bird flew .", "the bird ate seeds .",
]

words = sorted({w for s in toy_sentences for w in s.split()})

def encode(sentence):
    return tokenizer(sentence)["input_ids"]

def decode(ids):
    return " ".join(tokenizer.convert_ids_to_tokens(ids))

examples = [encode(s) for s in toy_sentences]
for sentence, ids in zip(toy_sentences, examples):
    print(f"{sentence:<22} → {ids}")

print("\nTotal vocabulary size:", tokenizer.vocab_size)


In [ ]:
def n_params(module):
    return sum(p.numel() for p in module.parameters() if p.requires_grad)

def check_count(module, expected, name):
    actual = n_params(module)
    assert actual == expected, f"{name}: expected {expected:,}, got {actual:,}. Check dimensions and bias."
    print(f"OK: {name}: {actual:,} trainable parameters")

def check_causal(module, x, cutoff, name):
    module.eval()
    with torch.no_grad():
        x_changed = x.clone()
        x_changed[:, cutoff:, :] = torch.randn_like(x_changed[:, cutoff:, :])
        y_a, y_b = module(x), module(x_changed)
    torch.testing.assert_close(y_a[:, :cutoff], y_b[:, :cutoff], atol=1e-5, rtol=1e-5)
    print(f"OK: {name}: positions before {cutoff} ignore future inputs")


## Stage 1 — Single causal attention head

A full-width head uses four affine projections: Q, K, V, and output. With `C = 32`, Q, K and V have **no bias**, the output projection has one. With `C = 32`, the expected count is `3 × (32×32) + 1 × (32×32 + 32) = 4,128` parameters.

Fill every TODO, then run the next cell.


In [ ]:
class SingleHeadCausalAttention(nn.Module):
    def __init__(self, C, max_t):
        super().__init__()
        self.C = C
        # TODO: create self.query, self.key, self.value, self.out_proj.
        # Q, K, V: nn.Linear(C, C, bias=False); out_proj: nn.Linear(C, C, bias=True).
        raise NotImplementedError("Replace with your layer definitions.")

    def forward(self, x):
        # x: (B, T, C)
        # TODO: q, k, v projections, each (B, T, C).
        # TODO: scores = q @ k.transpose(-2, -1) / math.sqrt(self.C).
        # TODO: Boolean lower-triangular mask, shape (T, T), on x.device.
        # TODO: scores.masked_fill(~mask, float("-inf")) BEFORE softmax.
        # TODO: weights = softmax(scores, dim=-1); return out_proj(weights @ v).
        raise NotImplementedError("Implement the causal attention forward pass.")


In [ ]:
# Run only after Stage 1 is complete.
single = SingleHeadCausalAttention(C, MAX_T).to(DEVICE)
check_count(single, 4128, "SingleHeadCausalAttention")
x = torch.randn(2, 6, C, device=DEVICE)
y = single(x)
assert y.shape == x.shape and torch.isfinite(y).all()
check_causal(single, x, cutoff=3, name="SingleHeadCausalAttention")


## Stage 2 — Multi-head causal attention

Each of four heads projects `C → D`, where `D = 8`; concatenate heads and apply one `C → C` projection. The expected total remains **4,128** parameters: `4 × 3 × (32×8) + (32×32 + 32)`.


In [ ]:
class CausalAttentionHead(nn.Module):
    def __init__(self, C, D, max_t):
        super().__init__()
        self.D = D
        # TODO: query, key, value are nn.Linear(C, D, bias=False).
        raise NotImplementedError("Create C→D Q/K/V projections.")

    def forward(self, x):
        # x: (B, T, C); return (B, T, D).
        # TODO: repeat causal-score, mask, softmax, weighted-value logic.
        raise NotImplementedError("Implement one head.")

class MultiHeadCausalAttention(nn.Module):
    def __init__(self, C, H, max_t):
        super().__init__()
        assert C % H == 0
        self.C, self.H, self.D = C, H, C // H
        # TODO: self.heads = nn.ModuleList([...]) containing H heads.
        # TODO: self.out_proj = nn.Linear(C, C, bias=True).
        raise NotImplementedError("Create heads and output projection.")

    def forward(self, x):
        # TODO: call each head; concatenate their (B,T,D) outputs on dim=-1.
        # TODO: apply out_proj; return (B,T,C).
        raise NotImplementedError("Implement multi-head attention.")


In [ ]:
# Run only after Stage 2 is complete.
multi = MultiHeadCausalAttention(C, H, MAX_T).to(DEVICE)
check_count(multi, 4128, "MultiHeadCausalAttention")
x = torch.randn(2, 7, C, device=DEVICE)
assert multi(x).shape == x.shape
check_causal(multi, x, cutoff=4, name="MultiHeadCausalAttention")


## Stage 3 — Decoder block

Use pre-normalisation and residual connections:

    x = x + attention(LayerNorm(x))
    x = x + MLP(LayerNorm(x))

Expected count: multi-head attention `4,128` + two LayerNorm layers `128` + MLP `C → 4C → C`, `8,352` = **12,608**.


In [ ]:
class MLP(nn.Module):
    def __init__(self, C):
        super().__init__()
        # TODO: Linear(C, 4*C), GELU, Linear(4*C, C); linear layers use bias=True.
        raise NotImplementedError("Create the MLP.")

    def forward(self, x):
        # TODO: return (B, T, C).
        raise NotImplementedError("Implement MLP forward.")

class DecoderBlock(nn.Module):
    def __init__(self, C, H, max_t):
        super().__init__()
        # TODO: ln1, attn, ln2, mlp.
        raise NotImplementedError("Create DecoderBlock layers.")

    def forward(self, x):
        # TODO: pre-norm residual attention, then pre-norm residual MLP.
        raise NotImplementedError("Implement DecoderBlock forward.")


In [ ]:
# Run only after Stage 3 is complete.
block = DecoderBlock(C, H, MAX_T).to(DEVICE)
check_count(block, 12608, "DecoderBlock")
x = torch.randn(2, 8, C, device=DEVICE)
assert block(x).shape == x.shape
check_causal(block, x, cutoff=4, name="DecoderBlock")


## Stage 4 — Tiny decoder language model

Add token and positional embeddings, one decoder block, a final LayerNorm, and an untied vocabulary head. Expected parameters:

    token embedding V×C:          30,522 × 32 =   976,704
    position embedding MAX_T×C:                    512
    decoder block:                              12,608
    final LayerNorm:                                64
    language-model head C×V + V:             1,007,226
    total:                                   1,997,114


In [ ]:
class TinyDecoderLM(nn.Module):
    def __init__(self, V, C, H, max_t):
        super().__init__()
        self.max_t = max_t
        # TODO: token_embedding = nn.Embedding(V, C)
        # TODO: position_embedding = nn.Embedding(max_t, C)
        # TODO: block = DecoderBlock(C, H, max_t), ln_f = nn.LayerNorm(C)
        # TODO: lm_head = nn.Linear(C, V, bias=True)
        raise NotImplementedError("Create the tiny language model.")

    def forward(self, token_ids, targets=None):
        # token_ids: (B, T), T <= max_t.
        # TODO: add token and positional embeddings; run block, ln_f, lm_head.
        # TODO: logits must have shape (B, T, V).
        # TODO: when targets exists, use F.cross_entropy(logits.flatten(0,1), targets.flatten()).
        # Return logits, loss; set loss to None without targets.
        raise NotImplementedError("Implement TinyDecoderLM forward.")


In [ ]:
# Run only after Stage 4 is complete.
model = TinyDecoderLM(V, C, H, MAX_T).to(DEVICE)
check_count(model, 1_997_114, "TinyDecoderLM")
ids = torch.tensor([examples[0]], dtype=torch.long, device=DEVICE)
logits, loss = model(ids, targets=ids)
assert logits.shape == (1, ids.size(1), V)
assert loss.ndim == 0 and torch.isfinite(loss)
print(f"OK: logits: {tuple(logits.shape)}; initial loss: {loss.item():.3f}")


## Optional training and generation

For next-token learning, input tokens predict the shifted target:

    input:  <bos> the cat sat .
    target: the cat sat . <eos>

After implementation passes, add a small AdamW training loop and watch loss decrease on the toy corpus. Then generate by repeatedly feeding the last `MAX_T` tokens, selecting from the final-position logits, and appending the next token.


## Submission checklist

- Single head: **4,128** parameters; future positions cannot affect earlier outputs.
- Multi-head attention: **4,128** parameters; output shape remains `(B, T, C)`.
- Decoder block: **12,608** parameters; residual paths preserve causality.
- Tiny decoder LM: **1,997,114** parameters; logits have shape `(B, T, V)`.
- You can explain why masking after softmax is incorrect.


---

# End-to-end example — text to next-token probabilities

This final example connects the complete pipeline:

    text → token IDs → token embeddings + positional embeddings
         → decoder block / causal attention → logits → softmax probabilities

Use the prompt **"the cat"**. Before optional training, the probabilities are random because model weights start randomly. After training on the toy corpus, words such as `sat` or `ate` should become more plausible continuations.


In [ ]:
# TODO: run this cell after TinyDecoderLM passes the acceptance test.
prompt = "the cat"
prompt_ids = torch.tensor([encode(prompt)[:-1]], dtype=torch.long, device=DEVICE)

# 1. Tokenisation
print("text:", prompt)
print("tokens:", decode(prompt_ids[0]))
print("token IDs:", prompt_ids.tolist())

# 2. Embeddings
T = prompt_ids.size(1)
positions = torch.arange(T, device=DEVICE)
token_vectors = model.token_embedding(prompt_ids)
position_vectors = model.position_embedding(positions)[None, :, :]
input_vectors = token_vectors + position_vectors
print("token embeddings shape:", tuple(token_vectors.shape))
print("combined input shape:", tuple(input_vectors.shape))

# 3. Causal attention forward pass
model.eval()
with torch.no_grad():
    attention_output = model.block.attn(model.block.ln1(input_vectors))
    logits, _ = model(prompt_ids)
    probabilities = torch.softmax(logits[:, -1, :], dim=-1)

print("attention output shape:", tuple(attention_output.shape))

# 4. Next-token probabilities
top_probabilities, top_ids = probabilities.topk(k=5, dim=-1)
for probability, token_id in zip(top_probabilities[0], top_ids[0]):
    print(f"{decode([token_id]):>10}: {float(probability):.3%}")
